# Notebook 3 : SQL

In [ ]:
# Décommenter la ligne suivante pour installer ibis
# %pip install 'ibis-framework[sqlite]'

3. Sélectionner l'identifiant `id`, le nom `nom` et l'identifiant de la station la plus proche `id_proche_1` depuis la table `Topologie`.

In [ ]:
# requête qui permet d elister les tables
import sqlite3

import pandas as pd
import ibis

from ibis import _

ibis.options.interactive = True

query_tables = "SELECT name FROM sqlite_master WHERE type='table'"

## STAR

Nous considérons les données des stations de vélos en libre service [STAR](https://www.star.fr/) de Rennes Métropole. Une copie de la base SQLite est disponible dans le fichier `star.db`. Nous utilisons d'abord Pandas pour répondre aux questions, puis Ibis.

1. Se connecter à la base de données et afficher la liste des tables à l'aide de la fonction `read_sql` de Pandas et de la requête `query_tables`.

In [6]:
con = sqlite3.connect("../data/star.db")
df = pd.read_sql(query_tables, con)
df

,name
0,Topologie
1,Etat


2. Récupérer le contenu de la table `Etat` dans un dataframe et afficher la liste des variables disponibles. Même question pour la table `Topologie`.

In [ ]:
# peu recommandé car on extrait la totalité des tables, ici c'est pour l'exemple
df_etat = pd.read_sql("SELECT * FROM Etat", con)
df_etat.dtypes

id                            int64
nom                          object
latitude                    float64
longitude                   float64
etat                         object
nb_emplacements               int64
emplacements_disponibles      int64
velos_disponibles             int64
date                        float64
data                         object
dtype: object

In [12]:
df_topologie = pd.read_sql("SELECT * FROM Topologie", con)
df_topologie.dtypes

id                     int64
nom                   object
adresse_numero        object
adresse_voie          object
commune               object
latitude             float64
longitude            float64
id_correspondance    float64
mise_en_service      float64
nb_emplacements        int64
id_proche_1            int64
id_proche_2            int64
id_proche_3            int64
terminal_cb           object
dtype: object

In [ ]:
df_topologie[["id", "nom", "id_proche_1"]]


,id,nom,id_proche_1
0,1,République,2
1,2,Mairie,1
2,3,Champ Jacquet,2
3,10,Musée Beaux-Arts,12
4,12,TNB,10
...,...,...,...
78,62,Clemenceau,63
79,66,Bréquigny Piscine,65
80,69,Champs Manceaux,66
81,85,La Courrouze,20


4. Faire une jointure sur la table précédente pour créer une table qui contient la liste des stations avec l'identifiant, le nom et le nom de la station la plus proche associée à l'identifiant `id_proche_1`. Les variables utilisées comme clés sont différents, penser à utiliser les arguments `left_on` et `right_on` de la méthode `merge`.

In [26]:
(
    df_topologie[["id", "nom", "id_proche_1"]]
    .merge(df_topologie[["id", "nom"]], left_on="id_proche_1", right_on="id")
)

,id_x,nom_x,id_proche_1,id_y,nom_y
0,1,République,2,2,Mairie
1,2,Mairie,1,1,République
2,3,Champ Jacquet,2,2,Mairie
3,10,Musée Beaux-Arts,12,12,TNB
4,12,TNB,10,10,Musée Beaux-Arts
...,...,...,...,...,...
76,55,Préfecture,60,60,Cucillé
77,62,Clemenceau,63,63,Henri Fréville
78,69,Champs Manceaux,66,66,Bréquigny Piscine
79,85,La Courrouze,20,20,Pont de Nantes


5. Ajouter à la table précédente la distance entre la station et la station la plus proche.

In [ ]:
df = (
    df_topologie[["id", "nom", "id_proche_1", "latitude", "longitude"]]
    .merge(df_topologie[["id", "nom", "latitude", "longitude"]], left_on="id_proche_1", right_on="id")
)

df["distance"]= (df["latitude_x"]-df["latitude_y"])**2 +(df["longitude_x"]-df["longitude_y"])**2

df = (
    df.drop(columns=["id_proche_1", "id_y", "latitude_x", "latitude_y", "longitude_x", "longitude_y"])
    .rename(columns={"id_x": "id", "nom_x": "nom", "nom_y": "nom_proche"})
)

df

,id,nom,nom_proche,distance
0,1,République,Mairie,0.000003
1,2,Mairie,République,0.000003
2,3,Champ Jacquet,Mairie,0.000003
3,10,Musée Beaux-Arts,TNB,0.000004
4,12,TNB,Musée Beaux-Arts,0.000004
...,...,...,...,...
76,55,Préfecture,Cucillé,0.000017
77,62,Clemenceau,Henri Fréville,0.000036
78,69,Champs Manceaux,Bréquigny Piscine,0.000066
79,85,La Courrouze,Pont de Nantes,0.000120


6. Créer une table avec le nom des trois stations les plus proches du point GPS *(48.1179151,-1.7028661)* classées par ordre de distance et le nombre de vélos disponibles dans ces stations.

In [40]:
lat, lon = 48.1179151,-1.7028661

df = df_topologie[["id", "nom", "latitude", "longitude"]].copy()

df["distance"] = df.apply(
    lambda row: (row.latitude - lat)**2 + (row.longitude - lon)**2,
    axis =1
)

# écritude équivalente : df["distance"] = (df["latitude"] - lat)**2 + df["longitude"] - lon)**2

df = df.drop(columns=["latitude", "latitude"])

df.nsmallest(3,"distance").merge(df_etat[["id", "velos_disponibles"]], on="id")

,id,nom,longitude,distance,velos_disponibles
0,56,Berger,-1.705097,0.000008,10
1,52,Villejean-Université,-1.704122,0.000012,11
2,38,Marbeuf,-1.702077,0.000039,9


7. Reprendre les questions précédentes en utilisant le module `ibis`. Pour les jointures, utiliser la méthode `left_join`.

In [ ]:
# import ibis et connection SQlite
import ibis
ibis.options.interactive = True
con = ibis.sqlite.connect()
con = ibis.sqlite.connect("../data/star.db")

# récupérer les 2 tables
con.tables
etat = con.table("Etat")
topologie = con.table("Topologie")
etat
topologie

┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ id    ┃ nom                ┃ adresse_numero ┃ adresse_voie                               ┃ commune ┃ latitude  ┃ longitude ┃ id_correspondance ┃ mise_en_service ┃ nb_emplacements ┃ id_proche_1 ┃ id_proche_2 ┃ id_proche_3 ┃ terminal_cb ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ int64 │ string             │ string         │ string                                     │ string  │ float64   │ float64   │ int64             │ float64         │ int64           │ int64       │ int64       │ int64       │ string      │
├───────┼────────────────────┼────────────────┼────────────────────────────────────────────┼─────────┼───────────┼───────────┼───────────────────┼─────────────────┼─────────────────┼─────────────┼─────────────┼─────────────┼─────────────┤
│     1 │ République         │ 19             │ Quai Lamartine                             │ Rennes  │ 48.110026 │ -1.678037 │             15006 │         14417.0 │              30 │           2 │          11 │          10 │ true        │
│     2 │ Mairie             │ 12             │ Galerie du Théâtre                         │ Rennes  │ 48.111624 │ -1.678757 │              NULL │         14417.0 │              24 │           1 │           3 │           9 │ true        │
│     3 │ Champ Jacquet      │ 8              │ Rue du Champ Jacquet                       │ Rennes  │ 48.112764 │ -1.680062 │              NULL │         14417.0 │              24 │           2 │           5 │           1 │ false       │
│    10 │ Musée Beaux-Arts   │ 3              │ Avenue Jean Janvier                        │ Rennes  │ 48.109601 │ -1.674080 │              NULL │         14417.0 │              16 │          12 │           1 │          36 │ false       │
│    12 │ TNB                │ 1              │ Square de Kergus                           │ Rennes  │ 48.107748 │ -1.673711 │              NULL │         14417.0 │              28 │          10 │          16 │          11 │ false       │
│    14 │ Laënnec            │ 11             │ Boulevard René Théophile Hyacinthe Laënnec │ Rennes  │ 48.106847 │ -1.665817 │              NULL │         14417.0 │              16 │          35 │          15 │          12 │ false       │
│    17 │ Charles de Gaulle  │ 2              │ Cours des Alliés                           │ Rennes  │ 48.105111 │ -1.677119 │             15007 │         14417.0 │              24 │          16 │          18 │          11 │ true        │
│    20 │ Pont de Nantes     │ 32             │ Rue du Docteur Francis Joly                │ Rennes  │ 48.102015 │ -1.684015 │              NULL │         14417.0 │              20 │          43 │          18 │          70 │ false       │
│    22 │ Oberthur           │ 4              │ Rue Gutenberg                              │ Rennes  │ 48.113550 │ -1.661858 │              NULL │         14417.0 │              20 │          44 │          35 │          42 │ false       │
│    25 │ Office de Tourisme │ 2              │ Rue Le Bouteiller                          │ Rennes  │ 48.110294 │ -1.683106 │              NULL │         14417.0 │              10 │          24 │           1 │           2 │ true        │
│     … │ …                  │ …              │ …                                          │ …       │         … │         … │                 … │               … │               … │           … │           … │           … │ …           │
└───────┴────────────────────┴────────────────┴────────────────────────────────────────────┴─────────┴───────────┴───────────┴───────────────────┴─────────────────┴────────────

In [51]:
# 3. Sélectionner l'identifiant `id`, le nom `nom` et l'identifiant de la station la plus proche `id_proche_1` 
# depuis la table `Topologie`.
topo = topologie.select("id", "nom", "id_proche_1")
topo

┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ id    ┃ nom                ┃ id_proche_1 ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ int64 │ string             │ int64       │
├───────┼────────────────────┼─────────────┤
│     1 │ République         │           2 │
│     2 │ Mairie             │           1 │
│     3 │ Champ Jacquet      │           2 │
│    10 │ Musée Beaux-Arts   │          12 │
│    12 │ TNB                │          10 │
│    14 │ Laënnec            │          35 │
│    17 │ Charles de Gaulle  │          16 │
│    20 │ Pont de Nantes     │          43 │
│    22 │ Oberthur           │          44 │
│    25 │ Office de Tourisme │          24 │
│     … │ …                  │           … │
└───────┴────────────────────┴─────────────┘

In [56]:
# 4. Faire une jointure sur la table précédente pour créer une table qui contient la liste des stations avec l'identifiant, 
# le nom et le nom de la station la plus proche associée à l'identifiant `id_proche_1`. 
# Les variables utilisées comme clés sont différents, penser à utiliser les arguments `left_on` et `right_on` de la méthode `merge`.
(
    topo
    .left_join(topologie, topo.id_proche_1 == topologie.id) 
    .select(topo.id, topo.nom, nom_proche = topologie.nom)
)

┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ id    ┃ nom                ┃ nom_proche         ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ int64 │ string             │ string             │
├───────┼────────────────────┼────────────────────┤
│     1 │ République         │ Mairie             │
│     2 │ Mairie             │ République         │
│     3 │ Champ Jacquet      │ Mairie             │
│    10 │ Musée Beaux-Arts   │ TNB                │
│    12 │ TNB                │ Musée Beaux-Arts   │
│    14 │ Laënnec            │ Pont de Châteaudun │
│    17 │ Charles de Gaulle  │ Champs Libres      │
│    20 │ Pont de Nantes     │ Cité Judiciaire    │
│    22 │ Oberthur           │ Metz - Sévigné     │
│    25 │ Office de Tourisme │ Place de Bretagne  │
│     … │ …                  │ …                  │
└───────┴────────────────────┴────────────────────┘

In [60]:
######################################################
## Regarder correction pour les questions 5 à 7 
######################################################

8. (*Bonus*) Écrire des requêtes SQL pour obtenir les résultats demandés dans les questions 3 à 6. La fonction `to_sql` pourra être utilisée pour de l'aide.

In [ ]:
######################################################
## Regarder correction 
######################################################

## Musique

Le dépôt GitHub [lerocha/chinook-database](https://github.com/lerocha/chinook-database) met à disposition des bases de données de bibliothèques musicales. Une copie de la base SQLite est disponible dans le fichier `chinook.db`.

1. Utiliser le module `ibis` pour vous connecter à la base de données et explorer les tables formant le jeu de données pour le découvrir. En particulier, remarquer comment les tables `Playlist`, `PlaylistTrack` et `Track` sont liées entre elles.

2. Quelles sont les playlists qui contiennent le plus de pistes ?

3. Construire une table contenant les informations suivantes sur la playlist `Classical` : le titre de chaque piste ainsi que le titre de l'album dont cette piste est tirée.

4. (*Bonus*) Écrire une requête SQL donnant le résultat de la question précédente. La fonction `to_sql` pourra être utilisée pour de l'aide.